# Đề tài 4: Phát hiện tin giả tiếng Việt từ văn bản báo chí và mạng xã hội (Fake News Detection)

Notebook này được tối ưu hóa để chạy trên **Google Colab** sử dụng phần cứng **GPU (T4, L4, v.v.)** miễn phí của Google.

### Quy trình hoạt động mới:
1. **Mã nguồn (Source Code):** Được nhân bản trực tiếp bằng `git clone` từ kho GitHub của bạn lên Colab (không cần copy code thủ công).
2. **Kết quả huấn luyện (Results):** Tất cả tệp trọng số (`.pt`) và lịch sử chạy (`.json`) sẽ được tự động lưu trực tiếp vào **Google Drive** của bạn. Điều này đảm bảo dữ liệu không bị mất đi khi Colab ngắt kết nối.

## 1. Kết nối Google Drive và chuẩn bị phần cứng
Chạy ô dưới đây để mount tài khoản Google Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra phần cứng GPU được cấp phát
import torch
print("Có hỗ trợ GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU:", torch.cuda.get_device_name(0))

## 2. Clone mã nguồn từ GitHub
Nhập URL kho chứa GitHub của bạn dưới đây để clone mã nguồn của dự án về máy ảo Colab.

In [ ]:
# Thay đổi URL dưới đây bằng link GitHub của bạn
REPO_URL = "https://github.com/TranXuanDuc28/Fake-news-detection.git"

# Xóa thư mục cũ nếu có trước khi clone
import shutil
import os
folder_name = REPO_URL.split("/")[-1].replace(".git", "")
if os.path.exists(folder_name):
    shutil.rmtree(folder_name)

!git clone {REPO_URL}

# Di chuyển vào thư mục dự án vừa clone
%cd {folder_name}
!ls -la

## 3. Cài đặt các thư viện cần thiết

In [ ]:
!pip install transformers scikit-learn pandas numpy matplotlib seaborn
print("Cài đặt hoàn tất!")

## 4. Cấu hình thư mục lưu kết quả trên Google Drive
Chúng ta sẽ định nghĩa một thư mục lưu trên MyDrive của bạn (ví dụ: `FakeNewsDetection_Results/`).

In [ ]:
import os

# Tạo thư mục trên Google Drive để lưu checkpoints và kết quả
DRIVE_DIR = "/content/drive/MyDrive/FakeNewsDetection_Results"
DRIVE_MODELS = os.path.join(DRIVE_DIR, "models")
DRIVE_DATA = os.path.join(DRIVE_DIR, "data")

os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)

print(f"Thư mục lưu mô hình trên Drive: {DRIVE_MODELS}")
print(f"Thư mục lưu lịch sử/kết quả trên Drive: {DRIVE_DATA}")

## 5. Tải và gộp bộ dữ liệu bổ sung VFND & ReINTEL (ĐÃ ĐƯỢC TÍCH HỢP SẴN)
Bộ dữ liệu phụ `additional_train.csv` đã được tích hợp sẵn đầy đủ cả VFND và ReINTEL (hơn 4,500 mẫu) trực tiếp trong mã nguồn GitHub của bạn.

**⚠️ LƯU Ý QUAN TRỌNG:** Bạn **KHÔNG CẦN CHẠY** ô này. Nếu chạy, script sẽ tải lại bản VFND gốc và ghi đè làm mất đi phần dữ liệu ReINTEL đã gộp.

In [ ]:
# Chạy script tải và gộp dữ liệu ngoài
!python download_and_merge_vfnd.py

## 6. Chạy Tuning Sweep quét tham số siêu tối ưu
Quá trình này kiểm tra các giá trị Dropout, Learning Rate và Batch Size trên toàn bộ dữ liệu hoặc tập con tùy chọn. Tất cả checkpoints lưu vào Drive, kết quả json lưu trực tiếp vào Drive của bạn.


In [ ]:
# Chạy tuning bằng GPU với tập subset gồm 2000 mẫu để hoàn tất nhanh chóng
!python src/tune.py --save_dir {DRIVE_MODELS} --data_dir data --output {DRIVE_DATA}/tuning_results.json --epochs 3 --no_oversample --use_class_weights --additional_dataset both


## 7. Huấn luyện mô hình tối ưu cuối cùng trên toàn bộ dữ liệu
Sử dụng cấu hình cân bằng dữ liệu bằng trọng số lớp (**Class Weights**) thay vì nhân bản dữ liệu (Oversampling) để hạn chế tình trạng quá khớp (Overfitting) trên lớp tin giả và lưu trọng số tối ưu cùng lịch sử huấn luyện trực tiếp vào Drive.
**Lưu ý:** Quá trình huấn luyện sử dụng kỹ thuật **Early Stopping** (dừng sớm nếu F1 không tăng sau 5 epoch) và **ReduceLROnPlateau** (tự động giảm tốc độ học đi 2 lần nếu độ chính xác chững lại).

In [ ]:
print("--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH BiLSTM TỐI ƯU ---")
!python src/train.py --model lstm --epochs 15 --patience 5 --save_dir {DRIVE_MODELS} --data_dir data --history_file {DRIVE_DATA}/lstm_history.json --no_oversample --use_class_weights --additional_dataset both

print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH LSTM 1 CHIỀU TỐI ƯU ---")
!python src/train.py --model lstm_1d --epochs 15 --patience 5 --save_dir {DRIVE_MODELS} --data_dir data --history_file {DRIVE_DATA}/lstm_1d_history.json --no_oversample --use_class_weights --additional_dataset both

print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH TRANSFORMER TỐI ƯU ---")
!python src/train.py --model transformer --epochs 15 --patience 5 --save_dir {DRIVE_MODELS} --data_dir data --history_file {DRIVE_DATA}/transformer_history.json --no_oversample --use_class_weights --additional_dataset both


## 8. Đồng bộ hóa lịch sử huấn luyện
Gộp tệp lịch sử huấn luyện của cả 3 mô hình lại thành `final_training_history.json` lưu trên Google Drive.

In [ ]:
import json
import os

lstm_h_path = os.path.join(DRIVE_DATA, "lstm_history.json")
lstm_1d_h_path = os.path.join(DRIVE_DATA, "lstm_1d_history.json")
trans_h_path = os.path.join(DRIVE_DATA, "transformer_history.json")
final_h_path = os.path.join(DRIVE_DATA, "final_training_history.json")

try:
    combined_history = {}
    
    # 1. Load LSTM history
    if os.path.exists(lstm_h_path):
        with open(lstm_h_path) as f:
            combined_history["lstm"] = json.load(f)
            
    # 1.b Load LSTM 1D history
    if os.path.exists(lstm_1d_h_path):
        with open(lstm_1d_h_path) as f:
            combined_history["lstm_1d"] = json.load(f)
            
    # 2. Load Transformer history (with fallback to models/history.json)
    if os.path.exists(trans_h_path):
        with open(trans_h_path) as f:
            combined_history["transformer"] = json.load(f)
    elif os.path.exists(os.path.join(DRIVE_MODELS, "history.json")):
        with open(os.path.join(DRIVE_MODELS, "history.json")) as f:
            combined_history["transformer"] = json.load(f)
        
    with open(final_h_path, "w", encoding="utf-8") as f:
        json.dump(combined_history, f, indent=4)
        
    print("✅ Đã gộp và tạo thành công final_training_history.json trên Google Drive!")
except Exception as e:
    print(f"❌ Lỗi khi gộp lịch sử huấn luyện: {e}")

## 9. Vẽ biểu đồ Lịch sử huấn luyện (Training History)
Vẽ trực quan hóa đường cong suy giảm lỗi (Loss) và chỉ số Val F1 qua các epoch của cả 3 mô hình BiLSTM, LSTM 1D và Transformer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    with open(final_h_path) as f:
        hist = json.load(f)
        
    models_to_plot = [m for m in ["lstm", "lstm_1d", "transformer"] if m in hist]
    num_models = len(models_to_plot)
    
    if num_models > 0:
        fig, axes = plt.subplots(1, num_models, figsize=(8 * num_models, 5.5), squeeze=False)
        
        for idx, model_key in enumerate(models_to_plot):
            ax = axes[0, idx]
            model_data = hist[model_key]
            epochs = np.arange(1, len(model_data["train_loss"]) + 1)
            
            # Lấy tên hiển thị của mô hình
            if model_key == "lstm":
                model_name = "BiLSTM"
            elif model_key == "lstm_1d":
                model_name = "LSTM 1D"
            else:
                model_name = "Transformer"
                
            if model_key == "transformer":
                # Đọc tên model name từ checkpoint nếu có
                try:
                    checkpoint_file = os.path.join(DRIVE_MODELS, "best_transformer.pt")
                    if not os.path.exists(checkpoint_file):
                        checkpoint_file = os.path.join(DRIVE_MODELS, "best_transformer_phobert.pt")
                    if not os.path.exists(checkpoint_file):
                        checkpoint_file = os.path.join(DRIVE_MODELS, "best_transformer_distilbert.pt")
                        
                    checkpoint = torch.load(checkpoint_file, map_location="cpu")
                    backbone = checkpoint.get("hyperparameters", {}).get("transformer_model_name", "transformer")
                    model_name = backbone.split("/")[-1].upper()
                except:
                    pass
            
            # Vẽ đồ thị Train Loss và Val Loss
            ax.plot(epochs, model_data["train_loss"], label="Train Loss", color="#1f77b4", marker="o", linewidth=2)
            ax.plot(epochs, model_data["val_loss"], label="Val Loss", color="#ff7f0e", marker="s", linewidth=2)
            
            ax.set_title(f"Loss Curve - {model_name}", fontsize=14, fontweight="bold")
            ax.set_xlabel("Epochs", fontsize=12)
            ax.set_ylabel("Loss", fontsize=12)
            ax.legend(fontsize=11)
            ax.grid(True, linestyle="--", alpha=0.6)
            ax.set_xticks(epochs)
            
        plt.tight_layout()
        plt.show()
    else:
        print("⚠ Không tìm thấy dữ liệu lịch sử của mô hình nào.")
except Exception as e:
    print(f"❌ Lỗi vẽ đồ thị: {e}")

## 10. Hiển thị Classification Report & Confusion Matrix
Ô này sẽ tải trực tiếp kết quả dự đoán test set, in ra báo cáo phân loại chi tiết (Precision, Recall, F1 của Tin thật/Tin giả) và vẽ ma trận nhầm lẫn Confusion Matrix cho cả 3 mô hình.

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

for model_type in ["lstm", "lstm_1d", "transformer"]:
    results_path = os.path.join(DRIVE_MODELS, f"test_{model_type}_results.json")
    if os.path.exists(results_path):
        print("\n" + "="*65)
        print(f" 📊 BÁO CÁO PHÂN LOẠI CHI TIẾT CHO MÔ HÌNH: {model_type.upper()}")
        print("="*65)
        
        with open(results_path) as f:
            res = json.load(f)
            
        targets = res.get("targets", [])
        preds = res.get("predictions", [])
        
        if targets and preds:
            # 1. Classification report
            print("\nClassification Report:")
            print(classification_report(targets, preds, target_names=["Tin thật (Real)", "Tin giả (Fake)"]))
            
            # 2. Confusion Matrix Heatmap
            cm = confusion_matrix(targets, preds)
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                        xticklabels=["Real", "Fake"], 
                        yticklabels=["Real", "Fake"])
            plt.title(f"Confusion Matrix - {model_type.upper()}", fontsize=11, fontweight="bold")
            plt.ylabel("Nhãn thực tế (True Label)")
            plt.xlabel("Nhãn dự đoán (Predicted Label)")
            plt.show()
        else:
            print("⚠️ Không tìm thấy dữ liệu targets/predictions trong tệp kết quả.")
    else:
        print(f"⚠️ Không tìm thấy tệp kết quả test cho {model_type.upper()} tại {results_path}")